# Signal To Image Converter

In [1]:
import os
import datetime
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.figure import Figure
from matplotlib.backends.backend_agg import FigureCanvasAgg
from sklearn.model_selection import train_test_split

### 0. Load Dataset

In [2]:
labels = ['AF', 'N']

fs = 250

type_ = 'cnn' # pilih 'cnn' atau 'pure' untuk mengganti jenis data

In [3]:
dataset_folder = 'dataset/'
filenames = []
for filename in os.listdir(dataset_folder):
    if filename.find("_all") > -1 :
        filenames.append(filename)
        
filenames

['test_all-v2.csv',
 'test_all.csv',
 'test_all_Conv_AE.csv',
 'train_all-v2.csv',
 'train_all.csv',
 'train_all_Conv_AE.csv']

- read dataset

In [4]:
train_df = []
test_df = []

if type_ == 'cnn' :
    train_df = pd.read_csv(dataset_folder + "train_all_Conv_AE.csv", header=None)
    test_df = pd.read_csv(dataset_folder + "test_all_Conv_AE.csv", header=None)
elif type_ == 'pure' :
    train_df = pd.read_csv(dataset_folder + "train_all.csv", header=None)
    test_df = pd.read_csv(dataset_folder + "test_all.csv", header=None)

In [5]:
ecg_df = pd.concat([train_df, test_df])

train_df = []
test_df = []

In [6]:
ecg_df[600]=ecg_df[600].astype(int)
equilibre=ecg_df[600].value_counts()

print(equilibre)

1    15000
0    15000
Name: 600, dtype: int64


### 1. Dataset Augmentation

In [7]:
# # sampling and resampling dataset

# from sklearn.utils import resample
# n_samples = 30000 
# random_states = [123, 124]

# dfs = []

# for i in range(len(equilibre)):
#     dfs.append(ecg_df[ecg_df[600]==i])
#     dfs[i]=resample(dfs[i],replace=True,n_samples=n_samples,random_state=random_states[i])

# ecg_df=pd.concat(dfs)

In [8]:
# ecg_df[600]=ecg_df[600].astype(int)
# equilibre=ecg_df[600].value_counts()

# print(equilibre)

In [9]:
y = ecg_df.iloc[:,600].values
X = ecg_df.iloc[:,:600].values

### 2. Convert Signal Sequence to Image

In [10]:
def sequence_to_img(data):
    fig = Figure(figsize=(8, 8), dpi=28)
    canvas = FigureCanvasAgg(fig)

    ax = fig.add_subplot(111)
    ax.plot(data[0])
    ax.plot(data[1])
    ax.set_ylim(0,1)
    ax.axis('off')
    ax.margins(0)
    fig.tight_layout(pad=0)

    canvas.draw()
    buf = canvas.buffer_rgba()
    img = np.asarray(buf)[:, :, :3]
    return img

In [2]:
img = sequence_to_img(np.reshape(X[0], (2, 300)))
plt.imshow(img)

NameError: name 'sequence_to_img' is not defined

- transform signal to Image

In [13]:
images = []
n = int(len(X)//2)
print("----- Signal to Image -------")
for i in range(len(X[:n])):
    signal = np.reshape(X[i], (2, 300))
    img = sequence_to_img(signal)
    
    images.append(img)
    
    if i % 500 == 0:
        tm = datetime.datetime.now().strftime("%H:%M:%S")
        print('[%s] - finish processing %d sample' % (tm, i))

----- Signal to Image -------
[11:27:42] - finish processing 0 sample
[11:28:03] - finish processing 500 sample
[11:28:21] - finish processing 1000 sample
[11:28:41] - finish processing 1500 sample
[11:29:00] - finish processing 2000 sample
[11:29:21] - finish processing 2500 sample
[11:29:43] - finish processing 3000 sample
[11:30:04] - finish processing 3500 sample
[11:30:22] - finish processing 4000 sample
[11:30:36] - finish processing 4500 sample
[11:30:53] - finish processing 5000 sample
[11:31:12] - finish processing 5500 sample
[11:31:32] - finish processing 6000 sample
[11:31:47] - finish processing 6500 sample
[11:32:02] - finish processing 7000 sample
[11:32:17] - finish processing 7500 sample
[11:32:32] - finish processing 8000 sample
[11:32:49] - finish processing 8500 sample
[11:33:05] - finish processing 9000 sample
[11:33:26] - finish processing 9500 sample
[11:33:45] - finish processing 10000 sample
[11:34:03] - finish processing 10500 sample
[11:34:22] - finish proces

In [14]:
if len(images) == n :
    print("----- Signal to Image -------")
    for i in range(len(X[n:])):
        signal = np.reshape(X[i + n], (2, 300))
        img = sequence_to_img(signal)

        images.append(img)

        if (i + n) % 500 == 0:
            tm = datetime.datetime.now().strftime("%H:%M:%S")
            print('[%s] - finish processing %d sample' % (tm, i + n))
else :
    print("please run previous block!")

----- Signal to Image -------
[11:37:07] - finish processing 15000 sample
[11:37:29] - finish processing 15500 sample
[11:37:52] - finish processing 16000 sample
[11:38:16] - finish processing 16500 sample
[11:38:38] - finish processing 17000 sample
[11:39:01] - finish processing 17500 sample
[11:39:18] - finish processing 18000 sample
[11:39:41] - finish processing 18500 sample
[11:40:00] - finish processing 19000 sample
[11:40:20] - finish processing 19500 sample
[11:40:45] - finish processing 20000 sample
[11:41:14] - finish processing 20500 sample
[11:41:49] - finish processing 21000 sample
[11:42:30] - finish processing 21500 sample
[11:43:03] - finish processing 22000 sample
[11:43:33] - finish processing 22500 sample
[11:43:53] - finish processing 23000 sample
[11:44:14] - finish processing 23500 sample
[11:44:32] - finish processing 24000 sample
[11:44:59] - finish processing 24500 sample
[11:45:26] - finish processing 25000 sample
[11:45:52] - finish processing 25500 sample
[1

- save image and target data as compresed numpy array

In [15]:
#np.savez_compressed('dataset/X_image', X=images)

In [16]:
#np.savez_compressed('dataset/y_target', y=y)

In [17]:
part = 5
diff = int(len(X)//part)
for i in range(part):
    start = i*diff
    stop = (i+1)*diff
    print("Save compressed matrix : id %d to %d in X_image_%d.npz" % (start, stop, i))
    np.savez_compressed('dataset/X_image_%d' % i, X=images[start : stop])
    np.savez_compressed('dataset/y_target_%d' % i, y=y[start : stop])

Save compressed matrix : id 0 to 6000 in X_image_0.npz
Save compressed matrix : id 6000 to 12000 in X_image_1.npz
Save compressed matrix : id 12000 to 18000 in X_image_2.npz
Save compressed matrix : id 18000 to 24000 in X_image_3.npz
Save compressed matrix : id 24000 to 30000 in X_image_4.npz
